# 01 - Generate synthetic expense data

### This notebook creates the dataset the whole system is tested on: 100 reports, about 330 claims, 25% of them out of policy, with hidden answer-key labels.

In [28]:
import random
from datetime import date, timedelta
from pathlib import Path
from typing import Any, NamedTuple

import pandas as pd
import yaml


def load_policy(path: Path) -> dict[str, Any]:
    """Load the expense policy config from a YAML file."""
    if not path.exists():
        raise FileNotFoundError(f"Policy file not found: {path}")
    with open(path) as f:
        return yaml.safe_load(f)


ROOT = Path.cwd().parent
SEED = 42
OUT_OF_POLICY_RATE = 0.25
BORDERLINE_RATE = 0.05
WINDOW_START = date(2026, 5, 1)
WINDOW_END = date(2026, 10, 31)
TODAY = date(2026, 11, 12)
rng = random.Random(SEED)
POLICY = load_policy(ROOT / "config" / "policy.yaml")
CATEGORIES = POLICY["categories"]
HIGH_COST_CITIES = set(POLICY["city_tiers"]["high_cost"])

assert POLICY["version"] == "1.0"
assert CATEGORIES["Lodging"]["nightly_cap"]["standard"] == 250.0
assert len(HIGH_COST_CITIES) == 6
assert random.Random(SEED).random() == random.Random(SEED).random()
print(f"Policy v{POLICY['version']} loaded: {len(CATEGORIES)} categories")

Policy v1.0 loaded: 10 categories


In [29]:
EMPLOYEES_PATH = ROOT / "data" / "raw" / "employees.csv"
TRAVEL_PROB = {
    "Sales": 0.7,
    "Marketing": 0.5,
    "Engineering": 0.3,
    "Operations": 0.4,
    "Finance": 0.3,
}


def load_employees(path: Path) -> pd.DataFrame:
    """Load employee roster, keeping every column as a string."""
    if not path.exists():
        raise FileNotFoundError(f"Employees file not found: {path}")
    return pd.read_csv(path, dtype=str)


employees_df = load_employees(EMPLOYEES_PATH)

assert len(employees_df) == 10, f"Expected 10 employees, got {len(employees_df)}"
assert employees_df["employee_id"].is_unique, "Duplicate employee_id found"

unknown_depts = set(employees_df["department"]) - set(TRAVEL_PROB)
assert not unknown_depts, f"Departments missing from TRAVEL_PROB: {unknown_depts}"

valid_managers = set(employees_df["employee_id"]) | {"BOARD"}
bad_managers = set(employees_df["manager_id"]) - valid_managers
assert not bad_managers, f"Unknown manager_id values: {bad_managers}"

print(f"Loaded: {len(employees_df)} employees")
employees_df.head(3)

Loaded: 10 employees


,employee_id,employee_name,title,department,home_city,manager_id
0,E001,Priya Raman,Director of Engineering,Engineering,"Austin, TX",E010
1,E002,Kevin Ortiz,Senior Software Engineer,Engineering,"Austin, TX",E001
2,E003,Lena Brooks,ML Engineer,Engineering,"Denver, CO",E001


In [30]:
MILEAGE_RATES = [
    (
        date.fromisoformat(band["from"]),
        date.fromisoformat(band["to"]),
        band["rate"],
    )
    for band in CATEGORIES["Mileage"]["rates"]
]


def money(amount: float) -> float:
    """Round a dollar amount to cents."""
    return round(amount, 2)


def rand_date(start: date, end: date, rand: random.Random) -> date:
    """Return a random date between start and end, inclusive.

    Args:
        start: First allowed date.
        end: Last allowed date.
        rand: Seeded random generator to draw from.

    Returns:
        A date in the closed range [start, end].
    """
    if end < start:
        raise ValueError(f"end {end} is before start {start}")
    span_days = (end - start).days
    return start + timedelta(days=rand.randint(0, span_days))


def city_tier(city: str) -> str:
    """Return the lodging cap tier for a city (POL-2.2)."""
    return "high_cost" if city in HIGH_COST_CITIES else "standard"


def mileage_rate(expense_date: date) -> float:
    """Return the per-mile rate in force on the expense date (POL-2.6).
    Raises:
        ValueError: If no rate band in policy.yaml covers the date.
    """
    for start, end, rate in MILEAGE_RATES:
        if start <= expense_date <= end:
            return rate
    raise ValueError(f"No mileage rate covers {expense_date}")


test_rng = random.Random(0)
sample_dates = [rand_date(WINDOW_START, WINDOW_END, test_rng) for _ in range(1_000)]
assert min(sample_dates) >= WINDOW_START, "Date before window"
assert max(sample_dates) <= WINDOW_END, "Date after window"
assert rand_date(TODAY, TODAY, test_rng) == TODAY, "One-day range failed"

assert city_tier("New York, NY") == "high_cost"
assert city_tier("San Antonio, TX") == "standard"

assert mileage_rate(date(2026, 6, 30)) == 0.725, "Last day of old rate"
assert mileage_rate(date(2026, 7, 1)) == 0.76, "first day of new rate"
try:
    mileage_rate(date(2025, 12, 31))
except ValueError:
    pass
else:
    raise AssertionError("2025 date should have no mileage rate")

assert money(12.345678) == 12.35
print(
    f"Helpers OK dates{min(sample_dates)} to {max(sample_dates)}, "
    f"mileage {mileage_rate(WINDOW_START)} -> {mileage_rate(WINDOW_END)}"
)

Helpers OK dates2026-05-01 to 2026-10-31, mileage 0.725 -> 0.76


In [31]:
# 4a Lookup lists and the line-item template
CITIES = [
    "New York, NY",
    "San Francisco, CA",
    "Boston, MA",
    "Seattle, WA",
    "Washington, DC",
    "Los Angeles, CA",
    "Austin, TX",
    "Dallas, TX",
    "Chicago, IL",
    "Denver, CO",
    "Atlanta, GA",
    "Phoenix, AZ",
    "Nashville, TN",
    "San Antonio, TX",
]
CLIENTS = [
    "Acme Corp",
    "Globex",
    "Initech",
    "Umbrella Health",
    "Stark Logistics",
    "Wayne Financial",
    "Hooli",
    "Soylent Foods",
    "Cyberdyne Systems",
]
CLIENT_PEOPLE = [
    "Alex Morgan",
    "Jordan Lee",
    "Taylor Reed",
    "Casey Patel",
    "Riley Chen",
    "Morgan Diaz",
    "Avery Shah",
    "Quinn Foster",
    "Drew Kowalski",
    "Jamie Ito",
]
CONFERENCES = [
    "AWS re:Invent",
    "SaaStr Annual",
    "Data + AI Summit",
    "Dreamforce",
    "Content Marketing World",
    "PyData Global",
    "Gartner Data & Analytics Summit",
]
MEETING_TOPICS = [
    "renewal",
    "expansion",
    "pilot results",
    "Q4 roadmap",
    "contract terms",
]
HOME_OFFICE_ITEMS = [
    "External monitor",
    "Ergonomic chair",
    "Standing desk converter",
    "Docking station",
    "Webcam and headset",
    "Keyboard and mouse",
]
SUPPLY_ITEMS = [
    "Notebooks and pens",
    "Printer ink",
    "Whiteboard markers",
    "Shipping supplies",
    "Desk organizers",
]
RIDE_MERCHANTS = ["Uber", "Lyft", "Yellow Cab", "Airport Parking"]
RENTAL_MERCHANTS = ["Enterprise Rent-A-Car", "Hertz"]
MERCHANTS = {
    "Airfare": [
        "Delta Air Lines",
        "United Airlines",
        "American Airlines",
        "Southwest Airlines",
        "Alaska Airlines",
    ],
    "Lodging": [
        "Marriott",
        "Hilton",
        "Hyatt",
        "Courtyard by Marriott",
        "Hampton Inn",
        "Westin",
    ],
    "Meals": [
        "Starbucks",
        "Panera Bread",
        "Chipotle",
        "Hotel Restaurant",
        "Sweetgreen",
        "Local Diner",
    ],
    "Client Meals": [
        "Capital Grille",
        "Ruth's Chris Steak House",
        "Fogo de Chao",
        "Eddie V's",
        "Nobu",
        "The Mercer Kitchen",
    ],
    "Ground Transport": RIDE_MERCHANTS + RENTAL_MERCHANTS,
    "Conference": CONFERENCES,
    "Software": CATEGORIES["Software"]["approved_vendors"],
    "Home Office": ["Amazon", "Best Buy", "IKEA", "Staples", "Apple Store"],
    "Office Supplies": ["Staples", "Office Depot", "Amazon", "Target"],
}

RECEIPT_THRESHOLD = POLICY["submission"]["receipt_required_over"]
CLEAN_MAX_PCT = 1 - POLICY["escalation"]["near_cap_pct"] - 0.01


def blank_item(category: str, expense_date: date, city: str) -> dict[str, Any]:
    """Return an empty line item with every column set to a default.

    Every claim has the same 16 columns, even when a field does not
    apply (e.g. ``miles`` on  a hotel stay), so the final table is
    rectangular.
    """
    return {
        "expense_date": expense_date,
        "category": category,
        "merchant": "",
        "description": "",
        "business_purpose": "",
        "amount": 0.0,
        "city": city,
        "nights": None,
        "attendees": None,
        "attendee_names": "",
        "fare_class": "",
        "flight_hours": None,
        "rental_class": "",
        "miles": None,
        "preapproved": False,
        "receipt_attached": True,
    }


missing_merchants = set(CATEGORIES) - set(MERCHANTS)
assert missing_merchants == {"Mileage"}, f"No merchants for: {missing_merchants}"
assert HIGH_COST_CITIES <= set(CITIES), "A high-cost city is missing from CITIES"
assert round(CLEAN_MAX_PCT, 2) == 0.89, f"Unexpected CLEAN_MAX_PCT: {CLEAN_MAX_PCT}"
assert len(blank_item("Meals", TODAY, "Austin, TX")) == 16
print(f"Lookups OK: {len(CITIES)} cities, {len(MERCHANTS)} merchant lists")

Lookups OK: 14 cities, 9 merchant lists


In [32]:
# 4b make_clean_item
def make_clean_item(
    category: str,
    expense_date: date,
    city: str,
    purpose: str,
    rand: random.Random,
    nights: int = 1,
) -> dict[str, Any]:
    """Build one in-policy line item for a category (POL-2).

    Amounts stay below ``CLEAN_MAX_PCT`` of every cap so a clean claim
    never looks "near cap" (POL-5) and never breaks a hard rule.

    Args:
        category: A key of ``CATEGORIES`` in policy.yaml.
        expense_date: Date the expense happened.
        city: City where it happened (sets the lodging tier).
        purpose: Business purpose for the trip or report.
        rand: Seeded random generator to draw from.
        nights: Hotel nights, used only for Lodging.

    Returns:
        A dict with the 16 line-item columns filled in.
    """
    if category not in CATEGORIES:
        raise ValueError(f"Unknown category: {category}")
    item = blank_item(category, expense_date, city)
    item["business_purpose"] = purpose
    rules = CATEGORIES[category]

    match category:
        case "Airfare":
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["fare_class"] = "Economy"
            item["flight_hours"] = round(rand.uniform(1.2, 4.8), 1)
            item["amount"] = money(rand.uniform(180, 650))
            item["description"] = f"Round-trip economy airfare to {city}"
        case "Lodging":
            cap = rules["nightly_cap"][city_tier(city)]
            nightly = rand.uniform(0.55 * cap, CLEAN_MAX_PCT * cap)
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["nights"] = nights
            item["amount"] = money(nightly * nights)
            item["description"] = f"Hotel, {nights} night(s) at ${nightly:,.2f}/night"
        case "Meals":
            cap = rules["daily_per_diem"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["amount"] = money(rand.uniform(18, CLEAN_MAX_PCT * cap))
            item["description"] = "Traveler meals for the day"
        case "Client Meals":
            attendees = rand.randint(2, 5)
            client = rand.choice(CLIENTS)
            guests = rand.sample(CLIENT_PEOPLE, attendees - 1)
            per_person = rand.uniform(35, CLEAN_MAX_PCT * rules["per_person_cap"])
            names = [f"{guest} ({client})" for guest in guests] + ["employee"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["attendees"] = attendees
            item["attendee_names"] = "; ".join(names)
            item["amount"] = money(per_person * attendees)
            item["description"] = f"Client dinner with {client}, {attendees} attendees"
            topic = rand.choice(MEETING_TOPICS)
            item["business_purpose"] = f"Dinner with {client} team to discuss {topic}"
        case "Ground Transport":
            if rand.random() < 0.25:
                rental_class = rand.choice(rules["allowed_rental_classes"])
                item["merchant"] = rand.choice(RENTAL_MERCHANTS)
                item["rental_class"] = rental_class
                item["amount"] = money(rand.uniform(120, 380))
                item["description"] = f"{rental_class} rental car"
            else:
                item["merchant"] = rand.choice(RIDE_MERCHANTS)
                item["amount"] = money(rand.uniform(12, 85))
                item["description"] = "Ride to client site / airport"
        case "Mileage":
            miles = rand.randint(20, 180)
            rate = mileage_rate(expense_date)
            item["merchant"] = "Personal vehicle"
            item["miles"] = miles
            item["amount"] = money(miles * rate)
            item["description"] = f"{miles} business miles at ${rate}/mile"
            item["receipt_attached"] = False
        case "Conference":
            conference = rand.choice(CONFERENCES)
            cap = rules["per_event_cap"]
            item["merchant"] = conference
            item["preapproved"] = True
            item["amount"] = money(rand.uniform(300, CLEAN_MAX_PCT * cap))
            item["description"] = f"{conference} registration"
            item["business_purpose"] = (
                f"Attend {conference} for training and customer networking"
            )
        case "Software":
            vendor = rand.choice(MERCHANTS[category])
            item["merchant"] = vendor
            item["amount"] = money(rand.uniform(10, 300))
            item["description"] = f"{vendor} monthly subscription"
        case "Home Office":
            cap = rules["per_purchase_cap"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["amount"] = money(rand.uniform(40, CLEAN_MAX_PCT * cap))
            item["description"] = rand.choice(HOME_OFFICE_ITEMS)
        case "Office Supplies":
            cap = rules["per_purchase_cap"]
            item["merchant"] = rand.choice(MERCHANTS[category])
            item["amount"] = money(rand.uniform(8, CLEAN_MAX_PCT * cap))
            item["description"] = rand.choice(SUPPLY_ITEMS)

    # Pol-1.3: receipts always attached over the threshold; ~80% below it.
    if category != "Mileage":
        over_threshold = item["amount"] > RECEIPT_THRESHOLD
        item["receipt_attached"] = over_threshold or rand.random() < 0.8
    return item

In [33]:
# 4c Tests for the generator
def clean_cap(item: dict[str, Any]) -> tuple[float, float] | None:
    """Return (value, cap) for capped categories, else None. Test helper."""
    rules = CATEGORIES[item["category"]]
    match item["category"]:
        case "Lodging":
            nightly = item["amount"] / item["nights"]
            return nightly, rules["nightly_cap"][city_tier(item["city"])]
        case "Meals":
            return item["amount"], rules["daily_per_diem"]
        case "Client Meals":
            return item["amount"] / item["attendees"], rules["per_person_cap"]
        case "Conference":
            return item["amount"], rules["per_event_cap"]
        case "Home Office" | "Office Supplies":
            return item["amount"], rules["per_purchase_cap"]
    return None


test_rng = random.Random(1)
checked = 0
for category in CATEGORIES:
    for _ in range(200):
        city = test_rng.choice(CITIES)
        item = make_clean_item(
            category, date(2026, 8, 3), city, "Test", test_rng, nights=3
        )
        checked += 1
        assert item["amount"] > 0, item
        capped = clean_cap(item)
        if capped:
            value, cap = capped
            assert value < CLEAN_MAX_PCT * cap + 0.01, f"Near/over cap: {item}"
        if category != "Mileage" and item["amount"] > RECEIPT_THRESHOLD:
            assert item["receipt_attached"], f"Missing receipt {item}"

first = make_clean_item("Meals", TODAY, "Austin, TX", "Test", random.Random(5))
second = make_clean_item("Meals", TODAY, "Austin, TX", "Test", random.Random(5))
assert first == second, "Same seed should give the same item"

try:
    make_clean_item("Yachts", TODAY, "Austin, TX", "Test", random.Random(5))
except ValueError:
    pass
else:
    raise AssertionError("Unknown category should raise ValueError")

print(f"Clean items OK: {checked} generated, 0 near or over a cap")

Clean items OK: 2000 generated, 0 near or over a cap


In [34]:
# 5a Report line-item builders (travel vs. local)
LOCAL_POOLS = {
    "Sales": [
        "Client Meals",
        "Client Meals",
        "Mileage",
        "Ground Transport",
        "Office Supplies",
    ],
    "Marketing": [
        "Software",
        "Client Meals",
        "Office Supplies",
        "Home Office",
        "Mileage",
    ],
    "Engineering": ["Software", "Software", "Home Office", "Office Supplies"],
    "Operations": ["Mileage", "Office Supplies", "Home Office", "Ground Transport"],
    "Finance": ["Software", "Office Supplies", "Mileage", "Client Meals"],
}
CONFERENCE_DEPTS = {"Engineering", "Marketing"}
MAX_ITEMS = 5
LATEST_TRIP_START = WINDOW_END - timedelta(days=40)


class ReportDraft(NamedTuple):
    """The pieces of one report before IDs and labels are added."""

    title: str
    report_type: str  # "travel" or "local"
    items: list[dict[str, Any]]
    last_date: date  # latest date covered; submission is counted from here


def local_purpose(category: str, rand: random.Random) -> str:
    """Return a business purpose for a local (non-travel) expense."""
    match category:
        case "Mileage":
            return f"Drive to {rand.choice(CLIENTS)} office for meeting"
        case "Ground Transport":
            return f"Rideshare to {rand.choice(CLIENTS)} meeting"
        case "Software":
            return "Tool required for daily work"
        case "Home Office":
            return "Remote work setup"
        case "Office Supplies":
            return "Team office supplies"
        case "Client Meals":
            return ""  # make_clean_items writes its own purpose
        case _:
            raise ValueError(f"No local purpose for category: {category}")


def make_travel_items(dept: str, home_city: str, rand: random.Random) -> ReportDraft:
    """Build the clean line items for one business trip.

    Every trip has airfare and lodging. Conference fees, meals, client
    dinners and ground transport are added by chance. Trips start
    Monday to Wednesday, so they end on a weekday (a weekend stay is
    a planted violation later, never part of a clean trip).
    """
    dest = rand.choice([c for c in CITIES if c != home_city])
    start = rand_date(WINDOW_START, LATEST_TRIP_START, rand)
    while start.weekday() > 2:  # 0=Mon ... 2=Wed
        start += timedelta(days=1)
    nights = rand.randint(1, 3)  # 1 to 3 nights for a typical business trip
    end = start + timedelta(days=nights)

    is_conference = dept in CONFERENCE_DEPTS and rand.random() < 0.4
    if is_conference:
        purpose = f"Attend conference in {dest}"
    else:
        purpose = f"On-site visit with {rand.choice(CLIENTS)} in {dest}"

    items = [
        make_clean_item("Airfare", start, dest, purpose, rand),
        make_clean_item("Lodging", start, dest, purpose, rand, nights=nights),
    ]
    if is_conference:
        items.append(make_clean_item("Conference", start, dest, purpose, rand))
    for day in range(rand.randint(0, 2)):
        meal_date = start + timedelta(days=min(day, nights))
        items.append(make_clean_item("Meals", meal_date, dest, purpose, rand))
    if dept == "Sales" and rand.random() < 0.7:
        dinner_date = start + timedelta(days=rand.randint(0, nights))
        items.append(make_clean_item("Client Meals", dinner_date, dest, purpose, rand))
    if rand.random() < 0.5:
        items.append(make_clean_item("Ground Transport", start, dest, purpose, rand))

    return ReportDraft(purpose, "travel", items[:MAX_ITEMS], end)


def make_local_items(dept: str, home_city: str, rand: random.Random) -> ReportDraft:
    """Build 1-4 clean local expenses from the department's category pool."""
    period_end = rand_date(WINDOW_START + timedelta(days=25), WINDOW_END, rand)
    title = f"Local expenses through {period_end:%b %d, %Y}"
    items = []
    for _ in range(rand.randint(1, 4)):
        category = rand.choice(LOCAL_POOLS[dept])
        expense_date = period_end - timedelta(days=rand.randint(0, 25))
        purpose = local_purpose(category, rand)
        items.append(make_clean_item(category, expense_date, home_city, purpose, rand))
    last = max(item["expense_date"] for item in items)
    return ReportDraft(title, "local", items, last)


# --- checks ---
assert set(LOCAL_POOLS) == set(TRAVEL_PROB), "Pool/department mismatch"
assert all(c in CATEGORIES for pool in LOCAL_POOLS.values() for c in pool)

test_rng = random.Random(2)
home = "Austin, TX"
for dept in TRAVEL_PROB:
    for _ in range(200):
        trip = make_travel_items(dept, home, test_rng)
        cats = [i["category"] for i in trip.items]
        assert 2 <= len(trip.items) <= MAX_ITEMS
        assert cats[:2] == ["Airfare", "Lodging"]
        assert trip.items[0]["expense_date"].weekday() <= 2, "Trip must start Mon-Wed"
        assert all(
            WINDOW_START <= i["expense_date"] <= trip.last_date <= WINDOW_END
            for i in trip.items
        )
        assert all(i["city"] != home for i in trip.items)
        assert "Conference" not in cats or dept in CONFERENCE_DEPTS
        assert "Client Meals" not in cats or dept == "Sales"

        local = make_local_items(dept, home, test_rng)
        assert 1 <= len(local.items) <= 4
        assert all(i["city"] == home for i in local.items)
        assert all(i["category"] in LOCAL_POOLS[dept] for i in local.items)
        assert all(WINDOW_START <= i["expense_date"] <= WINDOW_END for i in local.items)
        assert local.last_date == max(i["expense_date"] for i in local.items)
        assert all(i["business_purpose"] for i in local.items), "Empty business purpose"

a = make_travel_items("Sales", home, random.Random(9))
b = make_travel_items("Sales", home, random.Random(9))
assert a == b, "Same seed should give the same trip"
print(f"Report builders OK: {len(TRAVEL_PROB) * 200 * 2} drafts checked")

Report builders OK: 2000 drafts checked


In [35]:
# 5b Build every report and claim (all clean; violations are planted in cell 6)
REPORTS_PER_EMPLOYEE = 10
CLEAN_LABELS = {
    "in_policy": True,
    "violation_type": "",
    "expected_decision": "approve",
    "review_reason": "",
    "is_borderline": False,
    "label_note": "",
}


def build_reports(
    employees: pd.DataFrame, rand: random.Random
) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Build 10 clean reports per employee and flatten them into claims.

    Each employee's department sets the chance a report is a trip
    (``TRAVEL_PROB``). Each report is submitted 2-20 days after the
    last date it covers, never after ``TODAY``. Every claim starts
    with the clean labels; cell 6 overwrites the ones it makes
    out of policy.

    Returns:
        (reports, claims): one dict per report, one dict per line item.
    """
    reports: list[dict[str, Any]] = []
    claims: list[dict[str, Any]] = []
    for emp in employees.itertuples(index=False):
        for n in range(1, REPORTS_PER_EMPLOYEE + 1):
            report_id = f"R-{emp.employee_id}-{n:02d}"
            if rand.random() < TRAVEL_PROB[emp.department]:
                draft = make_travel_items(emp.department, emp.home_city, rand)
            else:
                draft = make_local_items(emp.department, emp.home_city, rand)
            lag = timedelta(days=rand.randint(2, 20))
            submitted = min(draft.last_date + lag, TODAY)

            reports.append(
                {
                    "report_id": report_id,
                    "employee_id": emp.employee_id,
                    "report_title": draft.title,
                    "report_type": draft.report_type,
                    "submitted_date": submitted,
                }
            )
            for line_no, item in enumerate(draft.items, start=1):
                claims.append(
                    {
                        **item,
                        "report_id": report_id,
                        "employee_id": emp.employee_id,
                        "line_no": line_no,
                        "claim_id": f"{report_id}-L{line_no}",
                        "submitted_date": submitted,
                        **CLEAN_LABELS,
                    }
                )
    return reports, claims


rng = random.Random(SEED)  # reset, so rerunning this cell gives the same data
reports, claims = build_reports(employees_df, rng)
claims_df = pd.DataFrame(claims)


# --- checks ---
max_lag = POLICY["submission"]["max_days_after_expense"]
lag_days = (
    pd.to_datetime(claims_df["submitted_date"])
    - pd.to_datetime(claims_df["expense_date"])
).dt.days
line_counts = claims_df.groupby("report_id")["line_no"].agg(["count", "max"])

assert len(reports) == len(employees_df) * REPORTS_PER_EMPLOYEE
assert claims_df["claim_id"].is_unique, "Duplicate claim_id"
assert line_counts["count"].between(1, MAX_ITEMS).all(), "Report size out of range"
assert (line_counts["count"] == line_counts["max"]).all(), "line_no has gaps"
assert claims_df["expense_date"].between(WINDOW_START, WINDOW_END).all()
assert (claims_df["submitted_date"] <= TODAY).all(), "Submitted in the future"
assert lag_days.between(0, max_lag).all(), "Clean claim is late or pre-dated"
assert (
    claims_df["in_policy"].all() and (claims_df["expected_decision"] == "approve").all()
)
assert claims_df["business_purpose"].str.len().gt(0).all(), "Empty business purpose"
for item in claims:
    capped = clean_cap(item)
    if capped:
        value, cap = capped
        assert value < CLEAN_MAX_PCT * cap + 0.01, f"Near/over cap: {item['claim_id']}"

travel_share = sum(r["report_type"] == "travel" for r in reports) / len(reports)
high_value = (claims_df["amount"] > POLICY["escalation"]["high_value_threshold"]).sum()
print(f"Built {len(reports)} reports, {len(claims_df)} claims")
print(
    f"Travel reports: {travel_share:.0%} | lag {lag_days.min()}-{lag_days.max()} days"
)
print(f"Clean claims over $1,000 (will become 'review'): {high_value}")
claims_df["category"].value_counts()

Built 100 reports, 330 claims
Travel reports: 58% | lag 3-41 days
Clean claims over $1,000 (will become 'review'): 4


category
Airfare             58
Lodging             58
Meals               56
Software            38
Client Meals        31
Ground Transport    28
Office Supplies     20
Home Office         18
Conference          13
Mileage             10
Name: count, dtype: int64

In [36]:
# 6a Choose which claims break policy, and how
VIOLATION_WEIGHTS = {
    "over_cap": 0.22,
    "prohibited_category": 0.20,
    "missing_receipt": 0.14,
    "late_submission": 0.12,
    "duplicate": 0.10,
    "personal_extension": 0.08,
    "split_transaction": 0.07,
    "premium_class": 0.07,
}
TRAVEL_ONLY = {"personal_extension", "premium_class"}
LOCAL_ONLY = {"split_transaction"}


class PlannedViolation(NamedTuple):
    """Which rule a claim will break, and the clean line it pairs with."""

    violation_type: str
    partner: int | None  # index of the paired claim (duplicate/split), else None


def plan_violations(
    claims: list[dict[str, Any]],
    reports: list[dict[str, Any]],
    rand: random.Random,
) -> dict[int, PlannedViolation]:
    """Pick 25% of claims to break policy, and choose a violation for each.

    Types are drawn with ``VIOLATION_WEIGHTS``, limited to what makes
    sense for the claim: weekend extension and premium airfare only on
    trips; split purchases only on local reports; a duplicate must copy
    an *earlier* clean line on the same report; a split needs another
    clean line to become "part 1". Each partner is used once and is
    never itself a violation.


    Returns:
        ``{claim index: PlannedViolation}`` for every selected claim.
    """
    n_bad = round(OUT_OF_POLICY_RATE * len(claims))
    selected = set(rand.sample(range(len(claims)), n_bad))
    report_type = {r["report_id"]: r["report_type"] for r in reports}
    lines_by_report: dict[str, list[int]] = {}
    for idx, claim in enumerate(claims):
        lines_by_report.setdefault(claim["report_id"], []).append(idx)

    taken: set[int] = set()  # partners already used
    plan: dict[int, PlannedViolation] = {}
    for i in sorted(selected):
        report_id = claims[i]["report_id"]
        partners = [
            j
            for j in lines_by_report[report_id]
            if j != i and j not in selected and j not in taken
        ]
        earlier = [j for j in partners if j < i]

        options = dict(VIOLATION_WEIGHTS)
        blocked = LOCAL_ONLY if report_type[report_id] == "travel" else TRAVEL_ONLY
        for vtype in blocked:
            options.pop(vtype)
        if not earlier:
            options.pop("duplicate")
        if not partners:
            options.pop("split_transaction", None)

        vtype = rand.choices(list(options), weights=list(options.values()))[0]
        partner = None
        if vtype == "duplicate":
            partner = rand.choice(earlier)
        elif vtype == "split_transaction":
            partner = rand.choice(partners)
        if partner is not None:
            taken.add(partner)
        plan[i] = PlannedViolation(vtype, partner)
    return plan


violation_plan = plan_violations(claims, reports, rng)
partner_ids = {p.partner for p in violation_plan.values() if p.partner is not None}


# --- checks ---
assert set(VIOLATION_WEIGHTS) == set(POLICY["violation_codes"]), (
    "weights/codes mismatch"
)
assert abs(sum(VIOLATION_WEIGHTS.values()) - 1) < 1e-9, "weights must sum to 1"
assert len(violation_plan) == round(OUT_OF_POLICY_RATE * len(claims))

report_type = {r["report_id"]: r["report_type"] for r in reports}
for i, (vtype, partner) in violation_plan.items():
    rtype = report_type[claims[i]["report_id"]]
    assert vtype not in TRAVEL_ONLY or rtype == "travel", f"{vtype} on local report"
    assert vtype not in LOCAL_ONLY or rtype == "local", f"{vtype} on travel report"
    if partner is None:
        assert vtype not in {"duplicate", "split_transaction"}, (
            f"{vtype} has no partner"
        )
        continue
    assert claims[partner]["report_id"] == claims[i]["report_id"], (
        "Partner on other report"
    )
    assert partner not in violation_plan, "Partner is itself a violation"
    assert vtype != "duplicate" or partner < i, "Duplicate must copy an earlier line"

n_partnered = sum(p.partner is not None for p in violation_plan.values())
assert len(partner_ids) == n_partnered, "A partner was used twice"

planned = pd.Series([p.violation_type for p in violation_plan.values()])
print(
    f"Planned {len(violation_plan)} violations ({len(violation_plan) / len(claims):.1%})"
)
planned.value_counts()

Planned 82 violations (24.8%)


prohibited_category    19
over_cap               16
missing_receipt        14
late_submission        11
duplicate               7
premium_class           6
personal_extension      6
split_transaction       3
Name: count, dtype: int64

In [37]:
# 6b-1 Violation builders: over cap, prohibited item, missing receipt, late
ITEM_FIELDS = list(blank_item("Meals", TODAY, "Austin, TX"))  # the 16 line-item columns
PROHIBITED_CODES = {p["code"] for p in POLICY["prohibited"]}
MAX_SUBMIT_DAYS = POLICY["submission"]["max_days_after_expense"]
OVER_CAP_CATEGORIES = {
    "travel": ["Lodging", "Meals", "Client Meals", "Conference"],
    "local": ["Client Meals", "Home Office", "Office Supplies"],
}
RECEIPT_CATEGORIES = {
    "travel": ["Lodging", "Airfare"],
    "local": ["Client Meals", "Home Office"],
}
# (category, merchant, description, POL-3 code, (min$, max $))
PROHIBITED_TEMPLATES = [
    (
        "Meals",
        "Pinstripe Bar & Lounge",
        "Drinks with coworkers after work, no clients present",
        "ALCOHOL_NO_CLIENT",
        (45, 160),
    ),
    (
        "Home Office",
        "Best Buy",
        "AirPods Pro for personal commute",
        "PERSONAL_ITEM",
        (180, 260),
    ),
    (
        "Ground Transport",
        "City Parking Enforcement",
        "Parking ticket near client office",
        "FINE",
        (40, 120),
    ),
    (
        "Lodging",
        "Hotel Spa",
        "Spa massage during trip",
        "WELLNESS_ENTERTAINMENT",
        (90, 220),
    ),
    ("Lodging", "Marriott", "In-room movies", "WELLNESS_ENTERTAINMENT", (18, 40)),
    (
        "Airfare",
        "Delta Air Lines",
        "Airfare for spouse to join trip",
        "FAMILY_TRAVEL",
        (220, 560),
    ),
    (
        "Office Supplies",
        "Amazon",
        "Amazon gift card for team member birthday",
        "GIFT_CARD",
        (50, 100),
    ),
]
ViolationResult = tuple[dict[str, Any], str]  # (new line item, label_note)


def item_fields(claim: dict[str, Any]) -> dict[str, Any]:
    """Copy just the 16 line-item columns (no IDs, no labels)."""
    return {key: claim[key] for key in ITEM_FIELDS}


def make_over_cap(
    claim: dict[str, Any], report: dict[str, Any], rand: random.Random
) -> ViolationResult:
    """Replace the claim with an item 10-150% over its category cap (POL-2)."""
    category = rand.choice(OVER_CAP_CATEGORIES[report["report_type"]])
    nights = claim["nights"] or 2
    item = make_clean_item(
        category,
        claim["expense_date"],
        claim["city"],
        claim["business_purpose"],
        rand,
        nights=nights,
    )
    rules = CATEGORIES[category]
    match category:
        case "Lodging":
            cap = rules["nightly_cap"][city_tier(item["city"])]
            nightly = cap * rand.uniform(1.15, 1.6)
            item["amount"] = money(nightly * item["nights"])
            item["description"] = (
                f"Hotel, {item['nights']} night(s) at ${nightly:,.2f}/night"
            )
            note = f"nightly ${nightly:,.2f} > cap ${cap:,.2f}"
        case "Meals":
            cap = rules["daily_per_diem"]
            item["amount"] = money(cap * rand.uniform(1.3, 2.4))
            note = f"daily meals ${item['amount']:,.2f} > ${cap:,.2f}"
        case "Client Meals":
            cap = rules["per_person_cap"]
            per_person = cap * rand.uniform(1.2, 1.9)
            item["amount"] = money(per_person * item["attendees"])
            note = f"per person ${per_person:,.2f} > ${cap:,.2f}"
        case "Conference":
            cap = rules["per_event_cap"]
            item["amount"] = money(cap * rand.uniform(1.1, 1.6))
            note = f"${item['amount']:,.2f} > event cap ${cap:,.2f}"
        case "Home Office":
            cap = rules["per_purchase_cap"]
            item["amount"] = money(cap * rand.uniform(1.2, 2.5))
            item["description"] = rand.choice(
                [
                    "Premium ergonomic chair",
                    "Ultrawide 49-inch monitor",
                    "Motorized standing desk",
                ]
            )
            note = f"${item['amount']:,.2f} > ${cap:,.2f}"
        case "Office Supplies":
            cap = rules["per_purchase_cap"]
            item["amount"] = money(cap * rand.uniform(1.3, 2.5))
            note = f"${item['amount']:,.2f} > ${cap:,.2f}"
        case _:
            raise ValueError(f"No over-cap rule for {category}")
    item["receipt_attached"] = True  # break ONE rule only
    return item, note


def make_prohibited(
    claim: dict[str, Any], report: dict[str, Any], rand: random.Random
) -> ViolationResult:
    """Replace the claim with a POL-3 prohibited expense."""
    category, merchant, description, code, (low, high) = rand.choice(
        PROHIBITED_TEMPLATES
    )
    item = blank_item(category, claim["expense_date"], claim["city"])
    item["merchant"] = merchant
    item["description"] = description
    item["business_purpose"] = claim["business_purpose"] or report["report_title"]
    item["amount"] = money(rand.uniform(low, high))
    if category == "Airfare":
        item["fare_class"] = "Economy"
        item["flight_hours"] = round(rand.uniform(1.5, 4.5), 1)
    return item, code


def make_missing_receipt(
    claim: dict[str, Any], report: dict[str, Any], rand: random.Random
) -> ViolationResult:
    """Drop the receipt from an item over the threshold (POL-1.3).

    Keeps the original item when it qualifies; otherwise draws a new
    one from ``RECEIPT_CATEGORIES`` until it is over the threshold.
    """
    item = item_fields(claim)
    while item["category"] == "Mileage" or item["amount"] <= RECEIPT_THRESHOLD:
        category = rand.choice(RECEIPT_CATEGORIES[report["report_type"]])
        item = make_clean_item(
            category,
            claim["expense_date"],
            claim["city"],
            claim["business_purpose"],
            rand,
            nights=2,
        )
    item["receipt_attached"] = False
    return item, f"${item['amount']:,.2f} with no receipt"


def make_late(
    claim: dict[str, Any], report: dict[str, Any], rand: random.Random
) -> ViolationResult:
    """Backdate the expense 65-120 days before submission (POL-1.2)."""
    item = item_fields(claim)
    days_late = rand.randint(MAX_SUBMIT_DAYS + 5, 120)
    item["expense_date"] = claim["submitted_date"] - timedelta(days=days_late)
    if item["category"] == "Mileage":  # rate depends on the date (POL-2.6)
        rate = mileage_rate(item["expense_date"])
        item["amount"] = money(item["miles"] * rate)
        item["description"] = f"{item['miles']} business miles at ${rate}/mile"
    item["description"] += " (missed in earlier report)"
    return item, f"submitted {days_late} days after expense"


# --- checks: run every builder on every claim with a separate RNG ---
REPORTS_BY_ID = {r["report_id"]: r for r in reports}
test_rng = random.Random(6)
before = [dict(c) for c in claims]

for claim in claims:
    report = REPORTS_BY_ID[claim["report_id"]]
    for builder in (make_over_cap, make_prohibited, make_missing_receipt, make_late):
        item, _ = builder(claim, report, test_rng)
        assert list(item) == ITEM_FIELDS, f"{builder.__name__}: wrong columns"

    item, note = make_over_cap(claim, report, test_rng)
    value, cap = clean_cap(item)
    assert value > cap, f"Not over cap: {note}"
    assert item["category"] in OVER_CAP_CATEGORIES[report["report_type"]]

    item, note = make_prohibited(claim, report, test_rng)
    assert note in PROHIBITED_CODES, f"Unknown code {note}"
    assert item["business_purpose"], "Prohibited item has no purpose"

    item, note = make_missing_receipt(claim, report, test_rng)
    assert item["amount"] > RECEIPT_THRESHOLD and not item["receipt_attached"]
    assert item["category"] != "Mileage"

    item, note = make_late(claim, report, test_rng)
    days = (claim["submitted_date"] - item["expense_date"]).days
    assert days > MAX_SUBMIT_DAYS, f"Not late: {days} days"
    if item["category"] == "Mileage":
        assert item["amount"] == money(
            item["miles"] * mileage_rate(item["expense_date"])
        )


assert claims == before, "A builder changed the original claims"
assert {code for *_, code, _ in PROHIBITED_TEMPLATES} <= PROHIBITED_CODES
print(f"Violation builders OK: 4 builders x {len(claims)} claims, originals untouched")

Violation builders OK: 4 builders x 330 claims, originals untouched


In [38]:
# 6b-2 Paired/trip violations, then apply the whole plan
FARE_RULES = CATEGORIES["Airfare"]
SPLIT_ITEMS = [
    "Motorized standing desk",
    "Dual monitor setup",
    "Ergonomic workstation bundle",
]


def make_duplicate(partner: dict[str, Any]) -> ViolationResult:
    """Copy an earlier clean line exactly (POL-1.4)."""
    return item_fields(partner), f"duplicate of {partner['claim_id']}"


def make_split(
    claim: dict[str, Any], rand: random.Random
) -> tuple[dict[str, Any], dict[str, Any], str]:
    """Split one Home Office purchase into two lines, each under the cap (POL-1.5).

    Returns:
        (part_1, part_2, note). Part 1 replaces the partner line and stays
        "clean" on its own, part 2 is the violation.
    """
    cap = CATEGORIES["Home Office"]["per_purchase_cap"]
    amount_1 = money(rand.uniform(300, 0.86 * cap))
    amount_2 = money(rand.uniform(max(cap + 40 - amount_1, 150), 0.86 * cap))
    merchant = rand.choice(["Amazon", "Best Buy", "IKEA"])
    product = rand.choice(SPLIT_ITEMS)

    part_1 = blank_item("Home Office", claim["expense_date"], claim["city"])
    part_1["merchant"] = merchant
    part_1["description"] = f"{product} (part 1 of 2)"
    part_1["business_purpose"] = "Remote work setup"
    part_1["amount"] = amount_1
    part_2 = {**part_1, "description": f"{product} (part 2 of 2)", "amount": amount_2}
    note = f"total ${amount_1 + amount_2:,.2f} > ${cap:,.2f} split across two lines"
    return part_1, part_2, note


def make_personal_extension(
    claim: dict[str, Any], report: dict[str, Any], rand: random.Random
) -> ViolationResult:
    """Add 2 weekend hotel nights tacked onto a business trip (POL-3.7).


    Uses the weekend after the trip when it is before the submission
    date; otherwise the weekend before the trip.
    """
    expense_date = claim["expense_date"]
    saturday = expense_date + timedelta(days=(5 - expense_date.weekday()) % 7)
    when = "after the trip ended"
    if saturday > claim["submitted_date"]:
        saturday -= timedelta(days=7)
        when = "before the trip started"
    item = make_clean_item("Lodging", saturday, claim["city"], "", rand, nights=2)
    item["description"] = f"Hotel, stayed Sat-Sun {when} to see the city"
    item["business_purpose"] = "Extended stay"
    return item, f"weekend nights {when}"


def make_premium_class(
    claim: dict[str, Any], report: dict[str, Any], rand: random.Random
) -> ViolationResult:
    """Premium airfare that breaks POL-2.1 (short flight, no approval, or first)."""
    item = blank_item("Airfare", claim["expense_date"], claim["city"])
    min_hours = FARE_RULES["business_class_min_flight_hours"]
    if rand.random() < 0.5:
        item["fare_class"] = "Business"
        item["flight_hours"] = round(rand.uniform(2.0, min_hours - 0.5), 1)
        item["preapproved"] = True
        note = f"business class on {item['flight_hours']}h flight (< {min_hours}h)"
    else:
        item["fare_class"] = rand.choice(["Business", "First"])
        item["flight_hours"] = round(rand.uniform(min_hours + 0.5, 11), 1)
        item["preapproved"] = False
        if item["fare_class"] == "First":
            note = "first class is never reimbursed"
        else:
            note = "business class without pre-approval"
    item["merchant"] = rand.choice(MERCHANTS["Airfare"])
    item["amount"] = money(rand.uniform(1200, 4800))
    item["description"] = (
        f"Round-trip {item['fare_class'].lower()}-class airfare to {item['city']}"
    )
    item["business_purpose"] = claim["business_purpose"]
    return item, note


SINGLE_BUILDERS = {
    "over_cap": make_over_cap,
    "prohibited_category": make_prohibited,
    "missing_receipt": make_missing_receipt,
    "late_submission": make_late,
    "personal_extension": make_personal_extension,
    "premium_class": make_premium_class,
}


def violation_labels(violation_type: str, note: str) -> dict[str, Any]:
    """Answer-key labels for an out-of-policy claim."""
    return {
        "in_policy": False,
        "violation_type": violation_type,
        "expected_decision": "deny",
        "review_reason": "",
        "is_borderline": False,
        "label_note": note,
    }


def apply_violations(
    claims: list[dict[str, Any]],
    plan: dict[int, PlannedViolation],
    reports: list[dict[str, Any]],
    rand: random.Random,
) -> list[dict[str, Any]]:
    """Return a new claim list with every planned violation applied.

    IDs (report, claim, line, employee, submitted date) never change;
    only the 16 line-item fields and the labels are replaced.
    """
    reports_by_id = {r["report_id"]: r for r in reports}
    out = [dict(c) for c in claims]
    for i in sorted(plan):
        vtype, partner = plan[i]
        claim = out[i]
        match vtype:
            case "duplicate":
                item, note = make_duplicate(out[partner])
            case "split_transaction":
                part_1, item, note = make_split(claim, rand)
                out[partner].update(part_1)  # partner keeps its clean labels
                note += f" (other part: {out[partner]['claim_id']})"
            case _:
                builder = SINGLE_BUILDERS[vtype]
                item, note = builder(claim, reports_by_id[claim["report_id"]], rand)
        claim.update(item)
        claim.update(violation_labels(vtype, note))
    return out


labeled_claims = apply_violations(claims, violation_plan, reports, rng)
labeled_df = pd.DataFrame(labeled_claims)
bad = labeled_df[~labeled_df["in_policy"]]

# --- checks ---
assert [c["claim_id"] for c in labeled_claims] == [c["claim_id"] for c in claims]
assert all(list(c) == list(claims[0]) for c in labeled_claims), "Columns changed"
assert all(c["in_policy"] for c in claims), "Clean claims were modified"
assert len(bad) == len(violation_plan), "Wrong number of violations"
assert (
    bad["violation_type"].value_counts().to_dict() == planned.value_counts().to_dict()
)
assert bad["label_note"].str.len().gt(0).all(), "Violation without a note"

dup_key = ["employee_id", "expense_date", "merchant", "amount"]
is_dup = labeled_df.duplicated(dup_key, keep="first")
assert set(labeled_df.loc[is_dup, "claim_id"]) == set(
    bad.loc[bad["violation_type"] == "duplicate", "claim_id"]
), "Duplicate labels don't match what the data shows"

for i, (vtype, partner) in violation_plan.items():
    claim = labeled_claims[i]
    if vtype == "split_transaction":
        part_1 = labeled_claims[partner]
        cap = CATEGORIES["Home Office"]["per_purchase_cap"]
        assert part_1["in_policy"] and part_1["merchant"] == claim["merchant"]
        assert (
            max(part_1["amount"], claim["amount"])
            < cap
            < part_1["amount"] + claim["amount"]
        )
    elif vtype == "personal_extension":
        assert claim["expense_date"].weekday() == 5, "Extension must start on Saturday"
        assert claim["expense_date"] <= claim["submitted_date"], (
            "Expense after submission"
        )
    elif vtype == "premium_class":
        short = claim["flight_hours"] < FARE_RULES["business_class_min_flight_hours"]
        assert claim["fare_class"] == "First" or short or not claim["preapproved"]

print(f"Applied {len(bad)} violations ({len(bad) / len(labeled_df):.1%} of claims)")
labeled_df["expected_decision"].value_counts()

Applied 82 violations (24.8% of claims)


expected_decision
approve    248
deny        82
Name: count, dtype: int64

In [39]:
# 6c Borderline claims (near cap, vague purpose) and high-value review
NEAR_CAP_PCT = POLICY["escalation"]["near_cap_pct"]
HIGH_VALUE = POLICY["escalation"]["high_value_threshold"]
VAGUE_PURPOSES = ["Business", "Meeting", "Misc", "Work stuff", "Team"]
FLAT_CAPS = { # categories whose cap applies to the whole amount
    "Meals": CATEGORIES["Meals"]["daily_per_diem"],
    "Home Office": CATEGORIES["Home Office"]["per_purchase_cap"],
    "Office Supplies": CATEGORIES["Office Supplies"]["per_purchase_cap"]
}


def review_labels(reason: str, note: str, borderline: bool) -> dict[str, Any]:
    """Answer-dey labels for an in-policy claim a human must review (POL-5)."""
    return {
        "in_policy": True,
        "violation_type": "",
        "expected_decision": "review",
        "review_reason": reason,
        "is_borderline": borderline,
        "label_note": note,
    }

def make_borderline(claim: dict[str, Any], rand: random.Random) -> dict[str, Any]:
    """Return a copy of a clean claim pushed into a human-review zone.
    
    Capped categories usually move to 92-99.5% of their cap (POL-5.2);
    everything else gets a vague business purpose (POL-5.3).
    """
    new = dict(claim)
    category = new["category"]
    if category in FLAT_CAPS and rand.random() < 0.6:
        cap = FLAT_CAPS[category]
        new["amount"] = money(cap * rand.uniform(0.92, 0.995))
        new["receipt_attached"] = True # keep it clean on every other rule
        note = f"${new['amount']:,.2f} within {NEAR_CAP_PCT:.0%} of ${cap:,.2f}"
        new.update(review_labels("near_cap", note, borderline=True))
    elif category == "Lodging" and rand.random() < 0.6:
        cap = CATEGORIES["Lodging"]["nightly_cap"][city_tier(new["city"])]
        nightly = cap * rand.uniform(0.92, 0.995)
        new["amount"] = money(nightly * new["nights"])
        new["description"] = f"Hotel, {new['nights']} night(s) at ${nightly:,.2f}/night"
        note = f"nightly ${nightly:,.2f} within {NEAR_CAP_PCT:.0%} OF ${cap:,.2F}"
        new.update(review_labels("near_cap", note, borderline=True))
    else:
        new["business_purpose"] = rand.choice(VAGUE_PURPOSES)
        note = "business purpose too vague"
        new.update(review_labels("vague_purpose", note, borderline=True))
    return new


def add_review_cases(
        claims: list[dict[str, Any]], protected: set[int], rand: random.Random
) -> list[dict[str, Any]]:
    """Return a new claim list with borderline and high-value review labels.
    
    ``protected`` bolds partner lines of duplicates/splits; changing one
    would break the violation that points at it, so they are skipped.
    """
    out = [dict(c) for c in claims]
    candidates = [i for i, c in enumerate(out) if c["in_policy"] and i not in protected]
    n_borderline = round(BORDERLINE_RATE * len(out))
    for i in rand.sample(candidates, n_borderline):
        out[i] = make_borderline(out[i], rand)

    for claim in out: # POL-5.1: any in-policy item over $1,000 goes to a human
        if(
            claim["in_policy"]
            and claim["expected_decision"] =="approve"
            and claim["amount"] > HIGH_VALUE
        ):
            note = f"${claim['amount']:,.2f} > ${HIGH_VALUE:,.0f}"
            claim.update(review_labels("high_value", note, borderline=False))
    return out

final_claims = add_review_cases(labeled_claims, partner_ids, rng)
final_df = pd.DataFrame(final_claims)
review = final_df[final_df["expected_decision"] == "review"]

# --- checks ---
before_bad = labeled_df.loc[~labeled_df["in_policy"], "claim_id"]
after_bad = final_df.loc[~final_df["in_policy"], "claim_id"]
assert  before_bad.tolist() == after_bad.tolist(), "Violations were charged"
assert final_df["is_borderline"].sum() == round(BORDERLINE_RATE * len(final_df))
assert not final_df.loc[sorted(partner_ids), "is_borderline"].any(), "Partner changed"
assert list(final_df.columns) == list(labeled_df.columns), "Columns changed"

for claim in final_claims:
    if claim["review_reason"] == "near_cap":
        value, cap =clean_cap(claim)
        assert (1 - NEAR_CAP_PCT) * cap <= value < cap, f"Not near cap: {claim}"
    elif claim["review_reason"] == "vague_purpose":
        assert claim["business_purpose"] in VAGUE_PURPOSES
    if claim["in_policy"] and claim["amount"] > HIGH_VALUE:
        assert claim["expected_decision"] == "review", f"Missed: {claim['claim_id']}"

is_dup = final_df.duplicated(dup_key, keep="first") # still detectable?
assert set(final_df.loc[is_dup, "claim_id"]) == set(
    final_df.loc[final_df["violation_type"] == "duplicate", "claim_id"]
)

print(f"Out of policy: {(~final_df['in_policy']).mean():.1%} | review: {len(review)}")
print(final_df["expected_decision"].value_counts().to_dict())
review["review_reason"].value_counts()



Out of policy: 24.8% | review: 17
{'approve': 231, 'deny': 82, 'review': 17}


review_reason
vague_purpose    12
near_cap          4
high_value        1
Name: count, dtype: int64